# 01 — EDA (Exploratory Data Analysis)

**Objetivo:** 
- Que datos tengo?
- Que forma tienen? (filas, columnas, tipos)
- Que tan sucios estan? (nulos, duplicados, basura)
- Que patrones hay? (distribucion de valores, outliers)

**Input:** `data/A3 INVENTORY.xlsx` — archivo crudo del ERP (sistema de inventario de planta manufacturera)

---
## 1.1 — Cargar el archivo

El archivo tiene 3 hojas de Excel:
- `Inventory by Storeroom` — inventario completo
- `OPEN ORDER` — ordenes de compra abiertas
- `ACTUAL INVENTORY` — resumen (se ignora)

Carga las 2 hojas utiles en 2 DataFrames separados.

Pista: `pd.read_excel()` acepta el parametro `sheet_name`

In [67]:
import pandas as pd

In [68]:
PATH_FILE="../data/inventory_anon.xlsx"

In [69]:
sheets = pd.read_excel(PATH_FILE, sheet_name=["Inventory by Storeroom", "OPEN ORDER"])
sheets

{'Inventory by Storeroom':          Region Name Country Name Operation Name Plant Code Plant Name  \
 0      North America       Mexico           OP-1   PLCODE-1    PLANT-1   
 1      North America       Mexico           OP-1   PLCODE-1    PLANT-1   
 2      North America       Mexico           OP-1   PLCODE-1    PLANT-1   
 3      North America       Mexico           OP-1   PLCODE-1    PLANT-1   
 4      North America       Mexico           OP-1   PLCODE-1    PLANT-1   
 ...              ...          ...            ...        ...        ...   
 20042            NaN          NaN            NaN        NaN        NaN   
 20043            NaN          NaN            NaN        NaN        NaN   
 20044            NaN          NaN            NaN        NaN        NaN   
 20045            NaN          NaN            NaN        NaN        NaN   
 20046            NaN          NaN            NaN        NaN        NaN   
 
       Currency  Exchange Rate Blanket Number  Commodity Code  Commodity

In [70]:
inv = sheets["Inventory by Storeroom"]
orders = sheets["OPEN ORDER"]

---
## 1.2 — Dimensiones

Cuantas filas y columnas tiene cada hoja

In [71]:
print(inv.shape)
print(orders.shape)

(20047, 62)
(3813, 21)


---
## 1.3 — Nombres de columnas

Lista todas las columnas de cada hoja. 

Esto te dice que informacion esta disponible.

In [72]:
inv.columns.to_list()

['Region Name',
 'Country Name',
 'Operation Name',
 'Plant Code',
 'Plant Name',
 'Currency',
 'Exchange Rate',
 'Blanket Number',
 'Commodity Code',
 'Commodity',
 'Established Part Category',
 'Part Category',
 'Store Room',
 'Stock Location',
 'Item Creation Date',
 'Item Number',
 'Status',
 'UOM',
 'Item Description',
 '2016 Usage',
 '2017 Usage',
 '2018 Usage',
 'Avg12Mon Usage',
 'Avg6Mon Usage',
 'Avg3Mon Usage',
 'Month To Date Usage',
 'Last Used',
 'Last Recieved',
 'Balance On Hand',
 'Current Price',
 'Inventory Value',
 'Inventory Turns',
 'Item Type',
 'Order Methodology',
 'Optimum Number',
 'Critcal Spares',
 'Max Order Qty',
 'Min Order Qty',
 'Item Lead Time',
 'Feb 2018 Usage',
 'Mar 2018 Usage',
 'Apr 2018 Usage',
 'May 2018 Usage',
 'Jun 2018 Usage',
 'Jul 2018 Usage',
 'Aug 2018 Usage',
 'Sep 2018 Usage',
 'Oct 2018 Usage',
 'Nov 2018 Usage',
 'Dec 2018 Usage',
 'Jan 2019 Usage',
 'Status Code',
 'Manufacturer Code',
 'Manufacturer Name',
 'Manufacturer Part',
 

In [73]:
orders.columns.to_list()

['Plant Code',
 'Item Number',
 'Item Description',
 'UOM',
 'PO Number',
 'Issue Date',
 'Order Qty',
 'Open Qty',
 'Required Date ',
 'BOH',
 'Avg. Mon. Usage',
 'Price',
 'Supplier Code',
 'Supplier Name',
 'Analyst Code',
 'Annual Usage',
 'Last Used Date',
 'Last Received Date',
 'Critical',
 'Last Modified',
 'Vendor PO']

---
## 1.4 - Tipos de dato

Que tipo tiene cada columna? (numerico, texto, fecha)
Esto te dice si pandas interpreto bien los datos o hay que convertir algo.

In [74]:
pd.set_option("display.max_rows", 100)
inv.dtypes

Region Name                             str
Country Name                            str
Operation Name                          str
Plant Code                              str
Plant Name                              str
Currency                                str
Exchange Rate                       float64
Blanket Number                          str
Commodity Code                          str
Commodity                           float64
Established Part Category               str
Part Category                           str
Store Room                              str
Stock Location                          str
Item Creation Date           datetime64[us]
Item Number                             str
Status                                  str
UOM                                     str
Item Description                        str
2016 Usage                          float64
2017 Usage                          float64
2018 Usage                          float64
Avg12Mon Usage                  

In [75]:
orders.dtypes

Plant Code                       str
Item Number                      str
Item Description                 str
UOM                              str
PO Number                        str
Issue Date            datetime64[us]
Order Qty                      int64
Open Qty                       int64
Required Date         datetime64[us]
BOH                          float64
Avg. Mon. Usage              float64
Price                        float64
Supplier Code                    str
Supplier Name                    str
Analyst Code                 float64
Annual Usage                 float64
Last Used Date               float64
Last Received Date           float64
Critical                     float64
Last Modified         datetime64[us]
Vendor PO                        str
dtype: object

---
## 1.5 — Valores nulos

Critico del EDA. Saber:
- Que columnas tienen nulos y cuantos?
- Hay filas que son casi 100% nulas? (basura del Excel)
- Hay columnas que son 100% nulas? (se pueden eliminar)


In [76]:
nulls_per_column_inv_pct = inv.isnull().mean() * 100

# filtra solo  las que tinenen algun nulo y ordenar de mas nulos a menos
nulls_per_column_inv_pct[nulls_per_column_inv_pct > 0 ].sort_values(ascending=False)

Critcal Spares               100.000000
Unnamed: 60                  100.000000
Last Recieved                 75.821819
Last Used                     58.298000
Manufacturer Part             42.215793
Item Description              23.589565
Store Room                    22.731581
Manufacturer Code             20.940789
Manufacturer Name             20.930813
Blanket Number                20.756223
Stock Location                19.279693
Exchange Rate                 18.945478
Plant Code                    18.945478
Part Category                 18.945478
Established Part Category     18.945478
Commodity                     18.945478
Commodity Code                18.945478
Status                        18.945478
UOM                           18.945478
Item Number                   18.945478
Item Creation Date            18.945478
2017 Usage                    18.945478
2016 Usage                    18.945478
2018 Usage                    18.945478
Currency                      18.945478


- 100% nulas: Critcal Spares y Unnamed: 60 — son basura, se eliminan
- ~19% nulas: casi todas las columnas tienen exactamente 18.94% ,no es coincidencia.
- Significa que hay un grupo de filas (~19% del total) que son casi completamente vacías.Son filas basura del Excel. 

### Nulls por fila para hoja inv

In [77]:
nulls_per_row = inv.isnull().sum(axis=1) # suma por fila (cuantos nulls tiene cada fila)
print("filas con 50+ columnas nulas:", (nulls_per_row >= 50).sum()) # cuenta cuantas filas tiene 50 0 mas oclumasn nulas
print("total de filas:", len(inv))
print("Porcetaje", f"{(nulls_per_row >= 50).sum() / len(inv) * 100:.1f}%")

filas con 50+ columnas nulas: 3798
total de filas: 20047
Porcetaje 18.9%


### Nulls por fila para hoja orders

In [78]:
nulls_per_row_orders = orders.isnull().sum(axis=1)
print("filas con 50+ columnas vacias en orders:", (nulls_per_row_orders >= 50).sum() )
print("total de filas en hoja orders", len(orders))
print("Porcetaje de filas null en orders:", f"{(nulls_per_row_orders >= 50).sum() / len(orders) * 100:.2f}%")

filas con 50+ columnas vacias en orders: 0
total de filas en hoja orders 3813
Porcetaje de filas null en orders: 0.00%


In [79]:
ord_null_pct = orders.isnull().mean() * 100

# filtra solo  las que tinenen algun nulo y ordenar de mas nulos a menos
ord_null_pct[ord_null_pct > 0].sort_values(ascending=False)

Avg. Mon. Usage       100.000000
Critical              100.000000
Analyst Code          100.000000
Annual Usage          100.000000
Last Used Date        100.000000
Last Received Date    100.000000
Vendor PO               5.455022
PO Number               0.104904
Required Date           0.026226
dtype: float64

6 columnas 100% nulas en orders - esas se eliminan

---
## 1.6 - Estadisticas descriptivas

Para las columnas numericas clave,  min, max, mean, mediana.

Columnas clave del inventario:
- `Balance On Hand` - stock actual
- `Current Price` - precio unitario
- `Inventory Value` - valor total del item (BOH * precio)
- `Average Daily Usage` - consumo diario promedio


In [80]:
inv.head(1)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Unnamed: 60,Unnamed: 61
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR


In [81]:
inv[["Balance On Hand", "Current Price", "Inventory Value", "Average Daily Usage"]].describe()

,Balance On Hand,Current Price,Inventory Value,Average Daily Usage
count,16249.000000,16249.000000,16249.000000,16249.000000
mean,22.179702,1056.714709,563.135424,1.043301
std,824.644175,6606.762674,3091.038284,47.138494
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,4.130000,0.000000,0.000000
50%,0.000000,72.000000,0.000000,0.000000
75%,3.000000,545.660000,116.010000,0.002740
max,102000.000000,308588.410000,144408.950000,5587.945205


- Balance On Hand: la mediana (50%) es 0 - más de la mitad de los items no tienen stock. El 75% tiene 3 o menos unidades. Pero el max es 102,000 — hay outliers enormes.

- Current Price: va de $0 a $308,588. El item más caro vale más de 300 mil

- Inventory Value: la mediana es $0 - la mayoría del valor está concentrado en pocos items (por eso ABC Classification es importante)       

- Average Daily Usage: el 75% es 0.003 - casi nadie usa nada. La gran mayoría de items están parados (dead stock potencial) 

- Esto confirma el patron Pareto: pocos items = mucho dinero, muchos items = nada. 

---
## 1.7 - Primeras observaciones

primeras filas de cada hoja para entender como se ven los datos reales.



In [82]:
inv.head()

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Unnamed: 60,Unnamed: 61
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0004,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0005,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,NaN,EUR


In [83]:
orders.head()

,Plant Code,Item Number,Item Description,UOM,PO Number,Issue Date,Order Qty,Open Qty,Required Date,BOH,...,Price,Supplier Code,Supplier Name,Analyst Code,Annual Usage,Last Used Date,Last Received Date,Critical,Last Modified,Vendor PO
0,PLCODE-2,ITEM-8207,ITEM-DESC-8185,EA,NaN,2017-07-06 16:03:24.780,1,1,2017-07-06,38.0,...,32.59,SUP-CODE-001,SUPPLIER-001,NaN,NaN,NaN,NaN,NaN,2019-02-19 01:09:04.533,VPO-0001
1,PLCODE-2,ITEM-5094,ITEM-DESC-4443,EA,NaN,2018-10-11 15:33:29.090,41,41,2018-10-18,134.0,...,6.48,SUP-CODE-002,SUPPLIER-002,NaN,NaN,NaN,NaN,NaN,2019-02-19 01:09:04.533,VPO-0002
2,PLCODE-2,ITEM-0862,ITEM-DESC-0844,EA,NaN,2017-12-13 14:55:05.830,1,1,2017-12-20,0.0,...,10666.67,SUP-CODE-003,SUPPLIER-003,NaN,NaN,NaN,NaN,NaN,2019-02-19 01:09:04.533,VPO-0003
3,PLCODE-2,ITEM-2684,ITEM-DESC-2590,EA,NaN,2018-05-31 13:02:06.260,78,60,NaT,180.0,...,111.40,SUP-CODE-004,SUPPLIER-004,NaN,NaN,NaN,NaN,NaN,2019-02-19 01:09:04.533,VPO-0004
4,PLCODE-2,ITEM-9324,ITEM-DESC-7356,EA,PO-0001,2016-12-21 12:08:18.927,3,3,2017-01-18,0.0,...,673.00,SUP-CODE-005,SUPPLIER-005,NaN,NaN,NaN,NaN,NaN,2019-02-19 01:09:04.533,VPO-0005


In [84]:
print(f"Valor total de inventario: ${inv["Inventory Value"].sum():,.0f}")

Valor total de inventario: $9,150,387


---
## 1.8 - Conclusiones del EDA

  **Inventory:**                              
  - Filas totales: 20,047 — filas basura (50+ nulos): 3,798 (18.9%)                               
  - Columnas totales: 62 — columnas 100% nulas: 2 (Critcal Spares, Unnamed: 60)                   
  - Valor total del inventario: $ 9,150,387                                                         
  - La mediana de BOH es 0 — mas de la mitad de los items no tienen stock                         
  - El 75% de items tiene Average Daily Usage casi 0 — la mayoria esta parado                     
                                                                                                  
  **Orders:**                                                                                     
  - Filas: 3,813 — sin filas basura                                                               
  - Columnas 100% nulas: 6       
  - Fechas de ordenes: 2016-2018 — al momento del snapshot (feb 2019), habia ordenes con 1-3 anios
   abiertas                                                                  